In [1]:
import os, json
import pandas as pd, numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
import joblib

In [2]:
df = pd.read_excel('/content/British Airways Summer Schedule Dataset - Forage Data Science Task 1.xlsx')

In [3]:
df.head(5)

,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX
0,2025-09-02,14:19:00,Afternoon,BA,BA5211,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,10,38
1,2025-06-10,06:42:00,Morning,BA,BA7282,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,7,28
2,2025-10-27,15:33:00,Afternoon,BA,BA1896,LHR,FRA,Germany,Europe,SHORT,A320,0,17,163,0,11,40
3,2025-06-15,18:29:00,Evening,BA,BA5497,LHR,IST,Turkey,Europe,SHORT,A320,0,8,172,0,16,54
4,2025-08-25,20:35:00,Evening,BA,BA1493,LHR,FRA,Germany,Europe,SHORT,A320,0,13,167,0,6,27


In [4]:
for col in ['FIRST_CLASS_SEATS','BUSINESS_CLASS_SEATS','ECONOMY_SEATS']:
  if col not in df:
    df[col] = 0
  df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)


In [5]:
df['total_seats'] = df['FIRST_CLASS_SEATS'] + df['BUSINESS_CLASS_SEATS'] + df['ECONOMY_SEATS']
df = df[df['total_seats'] > 0].copy()
print("Rows after total_seats>0 filter:", len(df))

Rows after total_seats>0 filter: 10000


In [6]:
for i in [1,2,3]:
    col = f'TIER{i}_ELIGIBLE_PAX'
    if col not in df.columns:
        df[col] = 0
    df[f'tier{i}_hist_pct'] = (df[col] / df['total_seats']).clip(0,1)

In [7]:
df['TIME_OF_DAY'] = df['TIME_OF_DAY'].astype(str)
df['ARRIVAL_REGION'] = df['ARRIVAL_REGION'].astype(str)
df['HAUL'] = df['HAUL'].astype(str)
df['AIRCRAFT_TYPE'] = df['AIRCRAFT_TYPE'].astype(str)
df['day_of_week'] = pd.to_datetime(df['FLIGHT_DATE']).dt.day_name()
df['month'] = pd.to_datetime(df['FLIGHT_DATE']).dt.month

In [8]:
cat_features = ['ARRIVAL_REGION','TIME_OF_DAY','HAUL','AIRCRAFT_TYPE','day_of_week']
X_cat = pd.get_dummies(df[cat_features].astype(str), drop_first=True)
X_num = df[['total_seats','month']].reset_index(drop=True)
X = pd.concat([X_num, X_cat.reset_index(drop=True)], axis=1)

In [9]:
cat_features = ['ARRIVAL_REGION','TIME_OF_DAY','HAUL','AIRCRAFT_TYPE','day_of_week']
X_cat = pd.get_dummies(df[cat_features].astype(str), drop_first=True)
X_num = df[['total_seats','month']].reset_index(drop=True)
X = pd.concat([X_num, X_cat.reset_index(drop=True)], axis=1)

In [10]:
y1 = df['tier1_hist_pct'].values
y2 = df['tier2_hist_pct'].values
y3 = df['tier3_hist_pct'].values

In [11]:
eps = 1e-6
def safe_logit(y):
    y = np.clip(y, eps, 1-eps)
    return np.log(y / (1-y))
def safe_expit(x):
    return 1 / (1 + np.exp(-x))

In [13]:
kf_splits = 5 # Define the number of splits for KFold
kf = KFold(n_splits=min(kf_splits, max(2, len(df))), shuffle=True, random_state=42)
scaler = StandardScaler()
X_scaled = X.copy()
if 'total_seats' in X_scaled.columns and 'month' in X_scaled.columns:
    X_scaled[['total_seats','month']] = scaler.fit_transform(X_scaled[['total_seats','month']])

def cv_eval(X_mat, y, model, transform=None):
    maes = []
    for train_idx, test_idx in kf.split(X_mat):
        X_tr, X_te = X_mat.iloc[train_idx], X_mat.iloc[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        if transform == 'logit':
            y_tr_t = safe_logit(y_tr)
            model.fit(X_tr, y_tr_t)
            pred_t = model.predict(X_te)
            pred = safe_expit(pred_t)
        else:
            model.fit(X_tr, y_tr)
            pred = model.predict(X_te)
            pred = np.clip(pred, 0, 1)
        maes.append(mean_absolute_error(y_te, pred))
    return float(np.mean(maes))

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

In [16]:
results = {}
models = {}

for tier, y in [('tier1', y1), ('tier2', y2), ('tier3', y3)]:
    print(f"\nTraining {tier} ...")
    ridge = Ridge(alpha=1.0)
    try:
        ridge_cv_mae = cv_eval(X_scaled, y, ridge, transform='logit')
    except Exception as e:
        ridge_cv_mae = None
        print("Ridge CV error:", e)
    try:
        ridge.fit(X_scaled, safe_logit(y))
    except Exception as e:
        print("Ridge fit error:", e)
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    try:
        rf_cv_mae = cv_eval(X, y, rf, transform=None)
    except Exception as e:
        rf_cv_mae = None
        print("RF CV error:", e)
    rf.fit(X, y)
    results[tier] = {'ridge_cv_mae':ridge_cv_mae, 'rf_cv_mae':rf_cv_mae}
    models[tier] = {'ridge':ridge, 'rf':rf}
    print(f"Results: ridge_cv_mae={ridge_cv_mae}, rf_cv_mae={rf_cv_mae}")

# Save model artefacts
joblib.dump(models['tier1']['ridge'], 'ridge_tier1.joblib')
joblib.dump(models['tier2']['ridge'], 'ridge_tier2.joblib')
joblib.dump(models['tier3']['ridge'], 'ridge_tier3.joblib')
joblib.dump(models['tier1']['rf'], 'rf_tier1.joblib')
joblib.dump(models['tier2']['rf'], 'rf_tier2.joblib')
joblib.dump(models['tier3']['rf'], 'rf_tier3.joblib')
print("\nSaved ridge_tier*.joblib and rf_tier*.joblib")


Training tier1 ...
Results: ridge_cv_mae=0.0029316484131779405, rf_cv_mae=0.0037150789830109983

Training tier2 ...
Results: ridge_cv_mae=0.028015875620715225, rf_cv_mae=0.021049862173004176

Training tier3 ...
Results: ridge_cv_mae=0.0704576444639761, rf_cv_mae=0.06208041719392032

Saved ridge_tier*.joblib and rf_tier*.joblib


In [17]:
# Empirical lookup with bounds (median, p25, p75, IQR, lower/upper)
group_cols = ['ARRIVAL_REGION','TIME_OF_DAY']
agg = df.groupby(group_cols).agg({
    'tier1_hist_pct':['median', lambda x: np.percentile(x,25), lambda x: np.percentile(x,75)],
    'tier2_hist_pct':['median', lambda x: np.percentile(x,25), lambda x: np.percentile(x,75)],
    'tier3_hist_pct':['median', lambda x: np.percentile(x,25), lambda x: np.percentile(x,75)],
    'total_seats':'median','FLIGHT_NO':'count'
})
agg.columns = ['_'.join([a,str(b)]) for a,b in agg.columns]
agg = agg.reset_index().rename(columns={
    'tier1_hist_pct_median':'Tier1_median_pct','tier1_hist_pct_<lambda_0>':'Tier1_p25','tier1_hist_pct_<lambda_1>':'Tier1_p75',
    'tier2_hist_pct_median':'Tier2_median_pct','tier2_hist_pct_<lambda_0>':'Tier2_p25','tier2_hist_pct_<lambda_1>':'Tier2_p75',
    'tier3_hist_pct_median':'Tier3_median_pct','tier3_hist_pct_<lambda_0>':'Tier3_p25','tier3_hist_pct_<lambda_1>':'Tier3_p75',
    'total_seats_median':'median_total_seats','FLIGHT_NO_count':'n_flights'
})
for t in ['Tier1','Tier2','Tier3']:
    agg[f'{t}_IQR'] = agg[f'{t}_p75'] - agg[f'{t}_p25']
    agg[f'{t}_lower'] = (agg[f'{t}_median_pct'] - agg[f'{t}_IQR']).clip(0,1)
    agg[f'{t}_upper'] = (agg[f'{t}_median_pct'] + agg[f'{t}_IQR']).clip(0,1)

agg.to_csv('empirical_lookup_with_bounds.csv', index=False)
print("Saved empirical_lookup_with_bounds.csv")

Saved empirical_lookup_with_bounds.csv


In [18]:
# Save JSON lookup
lookup_json = {}
for _, row in agg.iterrows():
    key = f"{row['ARRIVAL_REGION']}|{row['TIME_OF_DAY']}"
    lookup_json[key] = {
        'Tier1_median_pct': float(row['Tier1_median_pct']),
        'Tier1_p25': float(row['Tier1_p25']),
        'Tier1_p75': float(row['Tier1_p75']),
        'Tier1_IQR': float(row['Tier1_IQR']),
        'Tier1_lower': float(row['Tier1_lower']),
        'Tier1_upper': float(row['Tier1_upper']),
        'Tier2_median_pct': float(row['Tier2_median_pct']),
        'Tier2_p25': float(row['Tier2_p25']),
        'Tier2_p75': float(row['Tier2_p75']),
        'Tier2_IQR': float(row['Tier2_IQR']),
        'Tier2_lower': float(row['Tier2_lower']),
        'Tier2_upper': float(row['Tier2_upper']),
        'Tier3_median_pct': float(row['Tier3_median_pct']),
        'Tier3_p25': float(row['Tier3_p25']),
        'Tier3_p75': float(row['Tier3_p75']),
        'Tier3_IQR': float(row['Tier3_IQR']),
        'Tier3_lower': float(row['Tier3_lower']),
        'Tier3_upper': float(row['Tier3_upper']),
        'median_total_seats': int(row['median_total_seats']),
        'n_flights': int(row['n_flights'])
    }
with open('lookup_with_bounds.json','w') as f:
    json.dump(lookup_json, f, indent=4)
print("Saved lookup_with_bounds.json")

Saved lookup_with_bounds.json


In [27]:
# Apply median lookup to flights and compute estimated pax
def map_lookup_row(r):
    key = f"{r['ARRIVAL_REGION']}|{r['TIME_OF_DAY']}"
    if key in lookup_json:
        v = lookup_json[key]
        return pd.Series([v['Tier1_median_pct'], v['Tier2_median_pct'], v['Tier3_median_pct']])
    else:
        return pd.Series([df['tier1_hist_pct'].median(), df['tier2_hist_pct'].median(), df['tier3_hist_pct'].median()])

df[['Tier1_pct','Tier2_pct','Tier3_pct']] = df.apply(map_lookup_row, axis=1)
df['Tier1_est_pax'] = (df['total_seats'] * df['Tier1_pct']).round().astype(int)
df['Tier2_est_pax'] = (df['total_seats'] * df['Tier2_pct']).round().astype(int)
df['Tier3_est_pax'] = (df['total_seats'] * df['Tier3_pct']).round().astype(int)

df['category'] = df['ARRIVAL_REGION'] + '|' + df['TIME_OF_DAY']
final_df = df[['FLIGHT_NO','FLIGHT_TIME','ARRIVAL_STATION_CD','category','Tier1_pct','Tier2_pct','Tier3_pct','total_seats','Tier1_est_pax','Tier2_est_pax','Tier3_est_pax', 'ARRIVAL_REGION', 'TIME_OF_DAY', 'FIRST_CLASS_SEATS', 'BUSINESS_CLASS_SEATS', 'ECONOMY_SEATS', 'TIER1_ELIGIBLE_PAX', 'TIER2_ELIGIBLE_PAX', 'TIER3_ELIGIBLE_PAX']].copy()
final_df.to_csv('final_lookup_table.csv', index=False)
print("Saved final_lookup_table.csv")

Saved final_lookup_table.csv


In [21]:
# KMeans clustering
le_region = LabelEncoder().fit(df['ARRIVAL_REGION'])
le_time = LabelEncoder().fit(df['TIME_OF_DAY'])
cluster_features = pd.DataFrame({
    'total_seats': df['total_seats'].values,
    'region_code': le_region.transform(df['ARRIVAL_REGION']),
    'time_code': le_time.transform(df['TIME_OF_DAY'])
})
scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_features)
n_clusters = 4 # Define the number of clusters for KMeans
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(cluster_scaled)
df['cluster'] = clusters
cluster_summary = df.groupby('cluster').agg({
    'total_seats':'mean',
    'tier1_hist_pct':'mean',
    'tier2_hist_pct':'mean',
    'tier3_hist_pct':'mean',
    'FLIGHT_NO':'count'
}).rename(columns={'FLIGHT_NO':'n_flights'}).reset_index()
cluster_summary.to_csv('cluster_summary.csv', index=False)
df.to_csv('flights_with_clusters.csv', index=False)
print("Saved cluster_summary.csv and flights_with_clusters.csv")

Saved cluster_summary.csv and flights_with_clusters.csv


In [22]:
# Print final metrics
print("\n--- Done ---")
print("Flights analyzed:", len(df))
print("Lookup cells:", len(agg))
for tier in ['tier1','tier2','tier3']:
    print(tier, "Ridge CV MAE:", results[tier]['ridge_cv_mae'], "RF CV MAE:", results[tier]['rf_cv_mae'])
print("\nFiles saved in current working directory:")
print(" - final_lookup_table.csv")
print(" - empirical_lookup_with_bounds.csv")
print(" - lookup_with_bounds.json")
print(" - cluster_summary.csv")
print(" - flights_with_clusters.csv")
print(" - ridge_tier*.joblib (models)")
print(" - rf_tier*.joblib (models)")


--- Done ---
Flights analyzed: 10000
Lookup cells: 16
tier1 Ridge CV MAE: 0.0029316484131779405 RF CV MAE: 0.0037150789830109983
tier2 Ridge CV MAE: 0.028015875620715225 RF CV MAE: 0.021049862173004176
tier3 Ridge CV MAE: 0.0704576444639761 RF CV MAE: 0.06208041719392032

Files saved in current working directory:
 - final_lookup_table.csv
 - empirical_lookup_with_bounds.csv
 - lookup_with_bounds.json
 - cluster_summary.csv
 - flights_with_clusters.csv
 - ridge_tier*.joblib (models)
 - rf_tier*.joblib (models)


In [28]:
# Create a consistent grouping column in final_df
final_df['grouping'] = final_df['ARRIVAL_REGION'] + ' | ' + final_df['TIME_OF_DAY']

# Compute total seats
final_df['total_seats'] = final_df['FIRST_CLASS_SEATS'] + final_df['BUSINESS_CLASS_SEATS'] + final_df['ECONOMY_SEATS']

# Compute Tier percentages per flight
final_df['tier1_pct'] = final_df['TIER1_ELIGIBLE_PAX'] / final_df['total_seats']
final_df['tier2_pct'] = final_df['TIER2_ELIGIBLE_PAX'] / final_df['total_seats']
final_df['tier3_pct'] = final_df['TIER3_ELIGIBLE_PAX'] / final_df['total_seats']


In [29]:
# Group by our new grouping
lookup_group = final_df.groupby('grouping').agg({
    'tier1_pct':'median',
    'tier2_pct':'median',
    'tier3_pct':'median',
    'ARRIVAL_STATION_CD':'first'  # pick first destination as example
}).reset_index()

# Rename columns for clarity
lookup_group.rename(columns={
    'ARRIVAL_STATION_CD':'example_destination',
    'tier1_pct':'tier1 %',
    'tier2_pct':'tier2 %',
    'tier3_pct':'tier3 %'
}, inplace=True)

# Multiply by 100 to get percentages
lookup_group['tier1 %'] = (lookup_group['tier1 %']*100).round(1)
lookup_group['tier2 %'] = (lookup_group['tier2 %']*100).round(1)
lookup_group['tier3 %'] = (lookup_group['tier3 %']*100).round(1)

lookup_group.head()


,grouping,tier1 %,tier2 %,tier3 %,example_destination
0,Asia | Afternoon,0.0,3.0,11.1,HND
1,Asia | Evening,0.0,2.4,9.7,HND
2,Asia | Lunchtime,0.0,2.6,9.7,HND
3,Asia | Morning,0.0,2.5,9.8,HND
4,Europe | Afternoon,0.0,4.4,16.7,FRA


In [30]:
# Save the final lookup table for BA submission
lookup_group.to_excel('lounge_eligibility_lookup_table.xlsx', index=False)
print("lounge_eligibility_lookup_table.xlsx is ready")


lounge_eligibility_lookup_table.xlsx is ready


In [31]:
df_lounge = pd.read_excel('/content/lounge_eligibility_lookup_table.xlsx')

In [32]:
df_lounge.head()

,grouping,tier1 %,tier2 %,tier3 %,example_destination
0,Asia | Afternoon,0,3.0,11.1,HND
1,Asia | Evening,0,2.4,9.7,HND
2,Asia | Lunchtime,0,2.6,9.7,HND
3,Asia | Morning,0,2.5,9.8,HND
4,Europe | Afternoon,0,4.4,16.7,FRA
